# Part 2：通話とPDFから検索用データを作る
Part 1 のサンプルテーブルを使います。サンプル演習をスキップする場合は、`checkpoint_20261216.sql` の Part 1 と Part 2 を順に実行します。

## 1. 通話から問い合わせ種別と要約を抽出する
以下を SQL セルに貼り付けて実行します。抽出結果を Part 1 の文字起こしと比較してください。

```sql
SELECT AI_EXTRACT(text => transcript_text, responseFormat => {
  'inquiry_type': '問い合わせ種別を1つ。商品不具合、配送遅延、返品・交換、その他のいずれか',
  'summary': '通話の要約を60文字以内で記載'
}):response AS attributes
FROM P1_SAMPLE_CALL WHERE NULLIF(TRIM(transcript_text), '') IS NOT NULL;
```

## 2. PDFの表紙から属性を取り出し、検索用に本文を分割する
属性は検索結果を絞り込むため、チャンクは検索する本文の単位として使います。

```sql
SELECT AI_EXTRACT(text => doc:pages[0]:content::VARCHAR, responseFormat => {
  'title': '文書のタイトル',
  'doc_date': '発行日（YYYY-MM-DD）'
}):response AS attributes
FROM P1_SAMPLE_PDF WHERE doc:pages[0]:content IS NOT NULL;

SELECT page.index + 1 AS page_no, chunk.index + 1 AS chunk_no, chunk.value::VARCHAR AS chunk_text
FROM P1_SAMPLE_PDF, LATERAL FLATTEN(input => doc:pages) page,
LATERAL FLATTEN(input => SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(
  page.value:content::VARCHAR, 'markdown', 800, 100)) chunk
ORDER BY page_no, chunk_no;
```

モデルの抽出結果は変わることがあります。後半は同じデータで進めるため、`checkpoint_20261216.sql` の Part 2 を実行して全件の結果を復元してください。